In [1]:
import torch.nn as nn
import torch

In [2]:
class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)
    
    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**.5, dim=-1)

        context_vec = attn_weights @values
        return context_vec


In [3]:
inputs = torch.tensor([
    [.43,.15,.89],
    [.55,.87,.66],
    [.57,.85,.64],
    [.22,.58,.33],
    [.77,.25,.10],
    [.05,.80,.55]
])
d_in = inputs.shape[1] 
d_out = 2

In [4]:
torch.manual_seed(45)
sa_v2 = SelfAttention_v2(d_in,d_out)
print(sa_v2(inputs))


tensor([[-0.0246,  0.2314],
        [-0.0218,  0.2333],
        [-0.0220,  0.2332],
        [-0.0231,  0.2340],
        [-0.0272,  0.2320],
        [-0.0207,  0.2347]], grad_fn=<MmBackward0>)


In [5]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5,dim=1)
print(attn_weights)

tensor([[0.1726, 0.1604, 0.1608, 0.1685, 0.1735, 0.1642],
        [0.1821, 0.1682, 0.1684, 0.1571, 0.1689, 0.1553],
        [0.1813, 0.1678, 0.1681, 0.1578, 0.1690, 0.1560],
        [0.1775, 0.1698, 0.1699, 0.1583, 0.1666, 0.1580],
        [0.1634, 0.1605, 0.1607, 0.1735, 0.1710, 0.1708],
        [0.1854, 0.1733, 0.1735, 0.1512, 0.1651, 0.1515]],
       grad_fn=<SoftmaxBackward0>)


In [6]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length)) ##tril (Triangle Lower) = Lower Triangular matrix
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [7]:
##Will re-normalize attention weights to sum up to 1 again in each row.
## its acheived by dividing each element in each row by the sum in each row.
row_sums = mask_simple.sum(dim=1,keepdim=True)
masked_simple_norm = mask_simple/row_sums
print(masked_simple_norm)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667]])


In [8]:
##Now this works but there is data leakage because we have been using attention weights which is calculated by future weights.
##To overcome this will try to use upper triangular -ve infinity before doing softmax

mask = torch.triu(torch.ones(context_length,context_length),diagonal=1)
print(mask)

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])


In [9]:
mask = torch.triu(torch.ones(context_length,context_length),diagonal=1)
masked = attn_scores.masked_fill(mask.bool(),-torch.inf)
print(masked)

tensor([[-0.0769,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.2478,  0.1352,    -inf,    -inf,    -inf,    -inf],
        [ 0.2278,  0.1182,  0.1205,    -inf,    -inf,    -inf],
        [ 0.2252,  0.1625,  0.1635,  0.0634,    -inf,    -inf],
        [-0.1973, -0.2228, -0.2214, -0.1126, -0.1335,    -inf],
        [ 0.4237,  0.3283,  0.3295,  0.1352,  0.2591,  0.1377]],
       grad_fn=<MaskedFillBackward0>)


In [10]:
##Now will take this and apply softmax
attn_weights = torch.softmax(masked/keys.shape[-1]**.5,dim=1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5199, 0.4801, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3506, 0.3244, 0.3250, 0.0000, 0.0000, 0.0000],
        [0.2628, 0.2514, 0.2515, 0.2343, 0.0000, 0.0000],
        [0.1971, 0.1936, 0.1938, 0.2093, 0.2062, 0.0000],
        [0.1854, 0.1733, 0.1735, 0.1512, 0.1651, 0.1515]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
##Will try to apply dropout and try our causal self attention.
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6,6)
print(dropout(example)) ##on an average 50% will be set to 0, not every row 50% will be set to 0.

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [13]:
##Along with dropout will make sure the code to handle more than 1 input
## This ensures CausalAttention class supports the batch outputs produced by the dataloader we implemented earlier.

batch = torch.stack((inputs,inputs),dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [ ]:
class CausalAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in,d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = nn.Linear(d_in,d_out,bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask',torch.triu(torch.ones(context_length,context_length),diagonal=1))
        ##Register_buffer in pytorch is not strictly necessary for all use cases but it has advantages
        ## this will move to appropriate device CPU/GPU which will be relevant when training the LLM
    
    def forward(self,x):
        b, num_tokens, d_in = x.shape #batch size, number of tokens, input dim (vector embedding dim)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1,2) #here 1 means num_tokens, 2 means d_in 
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens],-torch.inf) #:num_tokens is to ensure for cases where number of tokens in the batch 
        ## is smaller than supported context_size
        attn_weights = torch.softmax(attn_scores/keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec


In [ ]:
torch.manual_seed(43)
context_length = batch.shape[1] ## it will take 6
ca = CausalAttention(d_in,d_out,context_length,0.0)  ## d_in=3 d_out=2
context_vecs = ca(batch)
print("context_vecs.shape:",context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


In [20]:
print(context_vecs)

tensor([[[ 0.0539, -0.1404],
         [ 0.0628, -0.1283],
         [ 0.0660, -0.1200],
         [ 0.0605, -0.1099],
         [ 0.0548, -0.0612],
         [ 0.0548, -0.0847]],

        [[ 0.0539, -0.1404],
         [ 0.0628, -0.1283],
         [ 0.0660, -0.1200],
         [ 0.0605, -0.1099],
         [ 0.0548, -0.0612],
         [ 0.0548, -0.0847]]], grad_fn=<UnsafeViewBackward0>)


In [ ]:
class MultiHeadAttentionWrapper(nn.Module):##d_out is 2 and final concatenated vector as output will be 4
    def __init__(self,d_in,d_out,context_length,dropout,num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in,d_out,context_length,dropout,qkv_bias)
             for _ in range(num_heads)
             ]
        )
    
    def forward(self,x):
        return  torch.cat([head(x) for head in self.heads],dim=-1)
    

In [22]:
inputs = torch.tensor([
    [.43,.15,.89],
    [.55,.87,.66],
    [.57,.85,.64],
    [.22,.58,.33],
    [.77,.25,.10],
    [.05,.80,.55]
])
batch = torch.stack((inputs,inputs),dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [26]:
torch.manual_seed(45)
context_length = batch.shape[1] #This is the number of tokens
d_in, d_out = 3,2
multiha =MultiHeadAttentionWrapper(d_in,d_out,context_length,0.0,num_heads=2)
context_vecs = multiha(batch)
print(context_vecs)
print("context vectors shape",context_vecs.shape)

tensor([[[ 0.2353,  0.2057,  0.3201, -0.3643],
         [ 0.0855,  0.2617,  0.3148, -0.3636],
         [ 0.0318,  0.2792,  0.3082, -0.3559],
         [ 0.0045,  0.2588,  0.2774, -0.3252],
         [-0.0226,  0.2245,  0.2114, -0.2282],
         [-0.0207,  0.2347,  0.2342, -0.2697]],

        [[ 0.2353,  0.2057,  0.3201, -0.3643],
         [ 0.0855,  0.2617,  0.3148, -0.3636],
         [ 0.0318,  0.2792,  0.3082, -0.3559],
         [ 0.0045,  0.2588,  0.2774, -0.3252],
         [-0.0226,  0.2245,  0.2114, -0.2282],
         [-0.0207,  0.2347,  0.2342, -0.2697]]], grad_fn=<CatBackward0>)
context vectors shape torch.Size([2, 6, 4])
